# How to Use Python with the ESPN Fantasy API

A practical reference for the ESPN Fantasy Football API surface this kind of project draws on -- draft history, projections, auction values, league settings, standings, matchups, weekly box scores, and transaction history -- covering how to call each one, what it returns, and the joins/mappings needed to turn opaque ids into something usable.

Two API eras matter throughout: seasons **2018+** are served by the *current-season* endpoint even after the year ends; seasons **2011-2017** only exist under the older *leagueHistory* endpoint (which wraps the payload in a one-element list) and are missing several data types entirely (no bench roster detail, no transaction/trade history, no player projections).

This notebook is about the API itself. For how the full production pipeline actually runs, see `README.md`.

## Authentication
Every call needs your league's cookies (required for private leagues) plus a real-looking `User-Agent` -- ESPN's API silently returns an empty or filtered response without one. Store credentials in a `.env` file (see README) rather than hardcoding them.

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
LEAGUE_ID = os.getenv('LEAGUE_ID')
espn_cookies = {
    'swid': os.getenv('SWID_COOKIE'),
    'espn_s2': os.getenv('ESPN_S2_COOKIES'),
}
HEADERS = {
    'Connection': 'keep-alive',
    'Accept': 'application/json, text/plain, */*',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.130 Safari/537.36',
}

## Two API hosts, two eras

- `https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/{season}/segments/0/leagues/{league_id}` -- works for **2018+**, including past completed seasons.
- `https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/leagueHistory/{league_id}?seasonId={season}` -- the only way to reach **2011-2017**; wraps the response in a one-element list.

Note: ESPN's API changed hosts in 2024 (`fantasy.espn.com` -> `lm-api-reads.fantasy.espn.com`); the URLs above are the current ones.

A `view` query param controls what's in the payload (`mDraftDetail`, `mSettings`, `mTeam`, `mMatchupScore`, `mBoxscore`, `mRoster`, ...) -- pass a list to combine several in one call. The reliable pattern used throughout this project: try the current-season endpoint first, fall back to leagueHistory on failure.

In [ ]:
def fetch_season_payload(season, params, extra_headers=None):
    headers = {**HEADERS, **(extra_headers or {})}
    url = f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/{season}/segments/0/leagues/{LEAGUE_ID}"
    r = requests.get(url, headers=headers, cookies=espn_cookies, params=params)
    if r.status_code == 200:
        return r.json()

    history_params = {**params, 'seasonId': season}
    r = requests.get(f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/leagueHistory/{LEAGUE_ID}",
                      headers=headers, cookies=espn_cookies, params=history_params)
    r.raise_for_status()
    data = r.json()
    return data[0] if isinstance(data, list) else data

**Pre-2018 gaps to plan around:** no full bench roster detail (only the 9 starters), no transaction/trade history of any kind, no player projections.

## Draft details (`mDraftDetail`)
One row per pick: overall pick number, `playerId`, `teamId` (the fantasy roster team, not the NFL team), the auction bid amount, and whether it was a kept player. `playerId`/`teamId` are opaque ids -- everything downstream joins on them.

In [ ]:
def get_draft_details(season):
    data = fetch_season_payload(season, {'view': ['mDraftDetail', 'mSettings', 'mTeam']})
    picks = pd.DataFrame(data['draftDetail']['picks'])
    return picks[['overallPickNumber', 'playerId', 'teamId', 'bidAmount', 'keeper']]

draft_df = get_draft_details(2024)
draft_df.sample(5)

## Player universe (`players_wl` view)
Every rosterable player's name, default position id, and real NFL team id. Needs its own `x-fantasy-filter`/`x-fantasy-platform` headers or ESPN returns a heavily filtered response -- these platform hash values were found by inspecting ESPN's own site network traffic, not documented anywhere; if they ever stop working, re-capture them from a browser's dev tools.

In [ ]:
def get_player_info(season):
    custom_headers = {
        'x-fantasy-filter': '{"filterActive":null}',
        'x-fantasy-platform': 'kona-PROD-1dc40132dc2070ef47881dc95b633e62cebc9913',
        'x-fantasy-source': 'kona',
    }
    url = f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/{season}/players?scoringPeriodId=0&view=players_wl"
    r = requests.get(url, cookies=espn_cookies, headers={**HEADERS, **custom_headers})
    df = pd.DataFrame(r.json())
    return df[['id', 'fullName', 'defaultPositionId', 'proTeamId']].rename(columns={'id': 'player_id'})

player_df = get_player_info(2024)
player_df.sample(5)

`defaultPositionId` needs mapping to a real position name:

In [ ]:
position_mapping = {1: 'QB', 2: 'RB', 3: 'WR', 4: 'TE', 5: 'K', 16: 'D/ST'}
player_df['position'] = player_df['defaultPositionId'].map(position_mapping)

## Real NFL team info (`proTeamSchedules_wl` view)
`proTeamId` on a player is the real NFL team, a completely different id space from the fantasy roster `teamId` above -- easy to conflate the two. This call resolves it to an abbreviation.

In [ ]:
def get_team_info(season):
    url = f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/{season}?view=proTeamSchedules_wl"
    r = requests.get(url, headers=HEADERS)
    teams = r.json()['settings']['proTeams']
    return pd.DataFrame(teams)[['id', 'abbrev']].rename(columns={'id': 'team_id'})

pro_team_df = get_team_info(2024)
pro_team_df.sample(5)

## Putting the draft together
Two inner joins turn the raw pick list into something readable: `playerId` -> player info, then `proTeamId` -> real NFL team. The fantasy roster `teamId` still needs a name -- either a static `{team_id: 'Owner Name'}` dict (fine for a single season with fixed ownership) or, if ownership changes hands across years, the GUID-based resolution covered later in this notebook.

In [ ]:
merged = draft_df.merge(player_df, left_on='playerId', right_on='player_id', how='inner')
merged = merged.merge(pro_team_df, left_on='proTeamId', right_on='team_id', how='inner')

league_teams = {1: 'Owner A', 2: 'Owner B', 3: 'Owner C'}  # placeholder -- build your own {team_id: name} dict, see static.py
merged['owner'] = merged['teamId'].map(league_teams)
merged[['overallPickNumber', 'fullName', 'position', 'abbrev', 'bidAmount', 'owner']].sample(5)

## Player projections & auction values (`kona_player_info` view)

Returns each player's ESPN-projected draft auction value, keeper value, and season point total/average -- but the season total is buried inside a `stats` array keyed by a stat id, not a plain field. The id convention is `'10' + season` (e.g. `'102024'` for the 2024 season's full-season projection) -- you have to know this to find the right entry.

`x-fantasy-filter` here does double duty: it also controls sort order and page size (350 covers the full rosterable pool).

In [ ]:
def get_player_projections(league_id, season):
    projection_year = str(season)
    headers = {
        'x-fantasy-filter': '{"players":{"filterSlotIds":{"value":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,23,24]},'
                             '"sortDraftRanks":{"sortPriority":2,"sortAsc":true,"value":"PPR"},"limit":350,"offset":0,'
                             '"filterRanksForScoringPeriodIds":{"value":[1]},"filterRanksForRankTypes":{"value":["PPR"]}}}',
        'x-fantasy-platform': 'kona-PROD-5cd7fbc8756f958a4250012b7badf69a8b3717d4',
        'x-fantasy-source': 'kona',
    }
    url = f"https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/{season}/segments/0/leagues/{league_id}?view=kona_player_info"
    r = requests.get(url, cookies=espn_cookies, headers={**HEADERS, **headers})
    players = r.json()['players']

    rows = []
    for entry in players:
        player = entry['player']
        for stat_line in player['stats']:
            if stat_line['id'] == '10' + projection_year:
                rows.append({
                    'player_id': player['id'],
                    'draftAuctionValue': entry['draftAuctionValue'],
                    'keeperValue': entry['keeperValue'],
                    'appliedTotal': stat_line['appliedTotal'],
                    'appliedAverage': stat_line['appliedAverage'],
                })
                break
    return pd.DataFrame(rows)

projections_df = get_player_projections(LEAGUE_ID, 2024)
projections_df.sample(5)

## League settings & teams (`mSettings` + `mTeam`)

One call covers a lot of ground: divisions, lineup slot counts (how many QB/RB/WR/... starting spots exist), playoff format, and every team's record/points/rank -- plus each team's `primaryOwner`, a GUID, and `members`, a GUID -> first/last name lookup.

Use `owner_guid`, not `teamId`, as the real "who owns this" identity if your league's team ids have ever changed hands across seasons (a new owner taking over an existing roster slot keeps the old `teamId`). `fetch_season_payload` (defined above) already handles the pre-2018 fallback.

In [ ]:
def get_league_settings_and_teams(season):
    data = fetch_season_payload(season, {'view': ['mSettings', 'mTeam']})
    settings = data['settings']
    schedule_settings = settings['scheduleSettings']
    roster_settings = settings['rosterSettings']

    divisions = {d['id']: d['name'] for d in schedule_settings.get('divisions', [])}
    lineup_slot_counts = {int(k): v for k, v in roster_settings['lineupSlotCounts'].items()}
    members = {m['id']: {'first_name': m.get('firstName', ''), 'last_name': m.get('lastName', '')}
               for m in data.get('members', [])}

    teams = []
    for team in data['teams']:
        record = team['record']['overall']
        owners = team.get('owners') or []
        teams.append({
            'team_id': team['id'],
            'division_id': team.get('divisionId'),
            'owner_guid': team.get('primaryOwner') or (owners[0] if owners else None),
            'team_name': team.get('name'),
            'wins': record.get('wins'), 'losses': record.get('losses'), 'ties': record.get('ties'),
            'points_for': record.get('pointsFor'), 'points_against': record.get('pointsAgainst'),
            'final_rank': team.get('rankCalculatedFinal'),
        })

    return {'divisions': divisions, 'lineup_slot_counts': lineup_slot_counts,
            'playoff_team_count': schedule_settings.get('playoffTeamCount'),
            'teams': teams, 'members': members}

settings = get_league_settings_and_teams(2024)
pd.DataFrame(settings['teams']).head()

## Resolving owner identity

`primaryOwner`/`members` gets you a GUID and a first/last name, but that's often incomplete or inconsistently cased, and doesn't handle two owners sharing a first name. The reliable approach: write `(season, team_id, owner_guid)` and `(guid, first_name, last_name)` out to two small CSVs, seed a display name for any new GUID automatically, then let a human rename entries by hand afterward (renaming never gets clobbered by a future refresh, since already-known GUIDs are never overwritten). Merge the two on `owner_guid`/`guid` wherever you need a real name.

In [ ]:
team_owners = pd.DataFrame([{'season': 2024, 'team_id': t['team_id'], 'owner_guid': t['owner_guid']} for t in settings['teams']])
members_df = pd.DataFrame([{'guid': g, **info} for g, info in settings['members'].items()])

# owner_names: a small, hand-editable guid -> display_name table you maintain yourself
owner_names = pd.DataFrame([{'guid': g, 'display_name': info['first_name']} for g, info in settings['members'].items()])

owner_lookup = team_owners.merge(owner_names, left_on='owner_guid', right_on='guid', how='left')
owner_lookup[['season', 'team_id', 'display_name']]

## Schedule / matchups (`mMatchupScore`)

One row per matchup; expand it to one row per team (with an `opponent`/`is_home` flip) to make weekly analysis easier. `playoffTierType` distinguishes regular season (`NONE`) from playoff brackets (`WINNERS_BRACKET` = real playoffs, anything else = a consolation bracket) -- useful for scoping stats to "regular season only" or finding who actually made the playoffs (more reliable than trusting `playoffSeed` directly, which doesn't always match real bracket participation).

In [ ]:
def get_schedule(season):
    data = fetch_season_payload(season, {'view': 'mMatchupScore'})
    rows = []
    for m in data.get('schedule', []):
        home, away = m.get('home'), m.get('away')
        if not home or not away:
            continue  # bye week
        for team, opp in [(home, away), (away, home)]:
            rows.append({
                'week': m['matchupPeriodId'], 'playoff_tier': m['playoffTierType'],
                'team_id': team['teamId'], 'opp_team_id': opp['teamId'],
                'team_points': team['totalPoints'], 'opp_points': opp['totalPoints'],
            })
    return pd.DataFrame(rows)

matchups_df = get_schedule(2024)
matchups_df.head()

## Weekly box scores (`mMatchupScore` + `mBoxscore`)

Per-player detail for a single week, both teams in the matchup: `lineup_slot`, actual `points`, and pre-game `projected_points`. The full roster (starters + bench) is under `rosterForCurrentScoringPeriod` -- only present for **2018+**; older seasons only expose `rosterForMatchupPeriod` (starters only, no reliable slot id), so check which key is present rather than assuming.

Gotcha: a `0.0` projection paired with a real nonzero actual score just means ESPN's projection engine hadn't populated that field yet when you queried early in the week -- not a real "projected for zero." A `0.0` projection alongside a `0.0` actual score is a normal bye/inactive bench player and should be left alone.

In [ ]:
NON_STARTER_SLOTS = {20, 21}  # Bench, IR

def get_week_boxscore(season, week):
    data = fetch_season_payload(season, {'view': ['mMatchupScore', 'mBoxscore'], 'scoringPeriodId': week})
    rows = []
    for m in data.get('schedule', []):
        if m['matchupPeriodId'] != week:
            continue
        for side in ('home', 'away'):
            team = m.get(side) or {}
            full_roster = team.get('rosterForCurrentScoringPeriod')
            has_bench_data = full_roster is not None
            roster = full_roster if has_bench_data else (team.get('rosterForMatchupPeriod') or {})

            for entry in roster.get('entries', []):
                player = entry['playerPoolEntry']['player']
                lineup_slot = entry.get('lineupSlotId') if has_bench_data else None
                points = entry['playerPoolEntry'].get('appliedStatTotal', 0.0)
                projected = next((s['appliedTotal'] for s in player['stats']
                                   if s.get('statSourceId') == 1 and s.get('scoringPeriodId') == week), None)
                if projected == 0 and points:
                    projected = None  # not populated yet, not a real zero
                rows.append({
                    'team_id': team.get('teamId'), 'player_id': entry.get('playerId'),
                    'player_name': player.get('fullName'), 'lineup_slot': lineup_slot,
                    'is_starter': (lineup_slot not in NON_STARTER_SLOTS) if has_bench_data else True,
                    'points': points, 'projected_points': projected, 'has_bench_data': has_bench_data,
                })
    return pd.DataFrame(rows)

boxscore_df = get_week_boxscore(2024, 1)
boxscore_df.head()

## Player transaction history (`kona_playercard` view)

A single view returns a given set of players' *complete* transaction history -- draft pick, every waiver/free-agent add and drop, every trade -- scoped via an `X-Fantasy-Filter` header listing the player ids you want (undocumented, found by inspecting the league-history page's own network traffic). One fetch covers all three uses below.

In [ ]:
import json as jsonlib

def fetch_playercard_transactions(season, player_ids):
    if not player_ids:
        return []
    headers = {'X-Fantasy-Filter': jsonlib.dumps({'players': {'filterIds': {'value': player_ids}}})}
    data = fetch_season_payload(season, {'view': 'kona_playercard', 'scoringPeriodId': 25}, extra_headers=headers)
    return data.get('players', [])

players_data = fetch_playercard_transactions(2024, [4362628, 4241389])  # example player ids

**Rookie detection:** a player has no real (`statSourceId == 0`) stats entry for the season *before* the one you're checking -- true rookies, and anyone ESPN's batched response omits entirely for that season, both count. Only reliable 2018+ (no usable stats at all in earlier seasons).

In [ ]:
def is_rookie(player, season):
    stats = player['player'].get('stats', [])
    has_real_stats = any(s.get('externalId') == str(season) and s.get('statSourceId') == 0 for s in stats)
    return not has_real_stats

# players_data above was fetched for `season` = the year BEFORE the draft year you're checking rookie status for

**Waiver/free-agent adds** and **trades** both live in the same `transactions` array per player, disambiguated by `type` and each item's `type`. A trade's items carry the exact `fromTeamId`/`toTeamId`/`playerId` straight from ESPN -- no roster-diff inference needed. Dedupe by the transaction's own `id` (or `relatedTransactionId` for trades), since the same event can appear under more than one involved player's history.

In [ ]:
def get_waiver_adds(players_data):
    seen, rows = set(), []
    for player in players_data:
        for t in player.get('transactions', []):
            if t.get('status') != 'EXECUTED' or t.get('type') not in ('WAIVER', 'FREEAGENT'):
                continue
            if t['id'] in seen:
                continue
            seen.add(t['id'])
            for item in t.get('items', []):
                if item.get('type') == 'ADD':
                    rows.append({'player_id': item['playerId'], 'team_id': item['toTeamId'],
                                 'week': t.get('scoringPeriodId'), 'bid_amount': t.get('bidAmount', 0)})
    return pd.DataFrame(rows)

def get_trades(players_data):
    seen, rows = set(), []
    for player in players_data:
        for t in player.get('transactions', []):
            if t.get('status') != 'EXECUTED':
                continue
            trade_id = t.get('relatedTransactionId') or t['id']
            for item in t.get('items', []):
                if item.get('type') != 'TRADE':
                    continue
                key = (trade_id, item['playerId'])
                if key in seen:
                    continue
                seen.add(key)
                rows.append({'trade_id': trade_id, 'player_id': item['playerId'],
                             'from_team_id': item['fromTeamId'], 'to_team_id': item['toTeamId'],
                             'week': t.get('scoringPeriodId')})
    return pd.DataFrame(rows)

Note: 2018 is a known gap for this specific trade path -- `kona_playercard` has no `TRADE`-type transactions that one season (every other type is present). A production fallback can replay each week's roster via `mTransactions2` and diff rosters before/after to infer trades when the direct data isn't there -- worth knowing if you build this out further, since trade-review setups where any team can cast a veto vote (not just the two actually involved) make a naive "who voted" signal unreliable.

## Current rosters (`mRoster` + `mTeam`)

A live snapshot of who's actually rostered right now, not reconstructed from draft/transaction history. `acquisitionType` (`DRAFT`/`TRADE`/`ADD`) is ESPN's own record of how each player joined their current team.

In [ ]:
def get_current_rosters(season):
    data = fetch_season_payload(season, {'view': ['mRoster', 'mTeam']})
    rows = []
    for team in data.get('teams', []):
        for entry in team.get('roster', {}).get('entries', []):
            player = entry['playerPoolEntry']['player']
            rows.append({
                'team_id': team['id'], 'player_id': entry['playerId'], 'fullName': player['fullName'],
                'acquisitionType': entry.get('acquisitionType'), 'lineupSlotId': entry.get('lineupSlotId'),
            })
    return pd.DataFrame(rows)

get_current_rosters(2024).head()

## Join/mapping cheat sheet

Every id that needs resolving before the raw API response is actually readable:

| id field | meaning | resolve via |
|---|---|---|
| `playerId` / `player_id` | a specific player | the universal join key across every endpoint above |
| `defaultPositionId` | fantasy position | `position_mapping` (`{1: 'QB', 2: 'RB', 3: 'WR', 4: 'TE', 5: 'K', 16: 'D/ST'}`) |
| `proTeamId` | real NFL team | `get_team_info` -- do not confuse with the fantasy roster `teamId` below |
| `teamId` (draft/roster/matchup) | fantasy roster team | `owner_guid` (via `mSettings`/`mTeam`) -> a hand-maintained display-name table, **not** a static `team_id` dict, if ownership has ever changed hands |
| `lineupSlotId` | starter vs. bench slot | `lineup_slot_mapping` (league-specific -- confirm against `rosterSettings.lineupSlotCounts`) |
| stat `id` on a projection/points entry | which season + period | `'10' + season` = that season's full-year total; `scoringPeriodId` = week number, `statSourceId` 0 = actual, 1 = projected |

## Where to go from here

Every function above is a trimmed teaching version. A real, battle-tested implementation -- with retries, edge-case handling (ambiguous trades, pre-2018 fallbacks, logo caching, optimal-lineup solving, and more) -- can go well beyond what's shown here; see `espn_api.py` for this project's own production version of the draft/player/projection calls. See `README.md` for how the full pipeline built on top of these actually runs.